# SparkClient: Live Cluster SparkConnect Demonstration

This notebook demonstrates how to provision, inspect, query, and manage interactive `SparkConnect` instances on Kubernetes using `SparkClient`.

### Prerequisites:
- A Kubernetes cluster (such as KinD) with the Kubeflow Spark Operator and SparkConnect CRD installed.
- `kubectl` configured to reach the target cluster.
- `pyspark` installed in the local environment (`uv pip install pyspark`).

## 1. Imports and Cluster Health Check

Verify cluster connectivity, select the target namespace, and initialize `SparkClient`.

In [ ]:
import os
import subprocess
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Driver, Executor, Name, SparkClient

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "spark-test")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

# Check cluster access via kubectl
res = subprocess.run(
    ["kubectl", "get", "namespace", namespace],
    capture_output=True,
    text=True,
)

if res.returncode == 0:
    print(f"✓ Namespace '{namespace}' reachable on cluster.")
else:
    print(f"Warning: Cluster connectivity check returned: {res.stderr.strip()}")

## 2. Example 1: Create SparkConnect Session with Defaults

Launch an interactive session using default sizing. Once ready, inspect the assigned service URL and driver pod.

In [ ]:
session_name = f"demo-sparkconnect-{uuid.uuid4().hex[:6]}"
print(f"Launching SparkConnect session: {session_name}...")
spark = client.connect(options=[Name(session_name)], timeout=120)

# Discover the auto-generated session
session = client.get_session(name=session_name)
default_session_name = session.name if session else None

if default_session_name:
    info = client.get_session(default_session_name)
    print(f"Session Name: {info.name}")
    print(f"State:        {info.state.value}")
    print(f"Driver Pod:   {info.driver_pod_name}")
    print(f"Service Name: {info.service_name}")

spark.stop()
print("Local SparkSession disconnected.")

## 3. Example 2: Create Named Session with Custom Compute Specs

Define a custom driver and executor topology using `Driver` and `Executor` objects.

In [ ]:
custom_name = f"my-spark-session-{uuid.uuid4().hex[:6]}"

driver_spec = Driver(resources={"cpu": "1", "memory": "512m"})
executor_spec = Executor(
    num_instances=2,
    resources_per_executor={"cpu": "1", "memory": "512m"},
)

print(f"Creating custom session: {custom_name}...")
spark_custom = client.connect(
    options=[Name(custom_name)],
    driver=driver_spec,
    executor=executor_spec,
    timeout=120,
)

info_custom = client.get_session(custom_name)
print(f"Custom session created: {info_custom.name} (State: {info_custom.state.value})")

spark_custom.stop()
print("Custom session client disconnected.")

## 4. Example 3: List and Inspect Active Sessions

Retrieve the list of all active `SparkConnect` instances in the current namespace.

In [ ]:
active_sessions = client.list_sessions()
print(f"Total active sessions found: {len(active_sessions)}")
for s in active_sessions:
    print(f"  - Name: {s.name:<25} State: {s.state.value}")

## 5. Example 4: Retrieve Server Logs

Stream the most recent log lines from the session driver pod.

In [ ]:
target_session = custom_name
print(f"Fetching logs for: {target_session}...")
try:
    logs = list(client.get_session_logs(target_session))
    print("Recent logs (last 20 lines):")
    print("-" * 70)
    for line in logs[-20:]:
        print(line.rstrip())
    print("-" * 70)
except Exception as e:
    print(f"Could not retrieve logs: {e}")

## 6. Example 5: Clean Up Resources

Delete the provisioned sessions from the Kubernetes cluster.

In [ ]:
for name in [custom_name, default_session_name]:
    if name:
        try:
            print(f"Deleting session: {name}...")
            client.delete_session(name)
            print(f"✓ Deleted {name}")
        except Exception as e:
            print(f"Warning deleting {name}: {e}")

print("Cleanup complete.")